# IoT Monitoring and Anomaly Examples

This notebook shows a practical IoT telemetry pattern in Databricks with monitoring-oriented outputs.

The flow covers:

- landing sensor telemetry in bronze
- cleaning readings in silver
- checking device heartbeat gaps
- flagging anomaly candidates for downstream review

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

In [ ]:
bronze_table = "main.demo.iot_sensor_bronze_monitoring"
silver_table = "main.demo.iot_sensor_silver_monitoring"
heartbeat_table = "main.demo.iot_device_heartbeat_status"
anomaly_table = "main.demo.iot_anomaly_candidates"

sensor_events = [
    ("device_001", "plant_a", "temperature", 21.3, "2026-04-05 06:00:00", "celsius"),
    ("device_001", "plant_a", "temperature", 21.7, "2026-04-05 06:01:00", "celsius"),
    ("device_001", "plant_a", "temperature", 38.4, "2026-04-05 06:02:00", "celsius"),
    ("device_002", "plant_a", "temperature", 19.8, "2026-04-05 06:00:00", "celsius"),
    ("device_002", "plant_a", "temperature", 20.0, "2026-04-05 06:12:00", "celsius"),
    ("device_003", "plant_b", "vibration", 3.2, "2026-04-05 06:00:00", "mm_s"),
    ("device_003", "plant_b", "vibration", 9.1, "2026-04-05 06:01:00", "mm_s"),
    ("device_004", "plant_b", "temperature", None, "2026-04-05 06:00:00", "celsius")
]

bronze_df = spark.createDataFrame(
    sensor_events,
    ["device_id", "site_id", "sensor_type", "reading_value", "reading_ts_raw", "unit"]
)

## Bronze layer

Bronze stores raw telemetry with minimal assumptions and an ingest timestamp.

In [ ]:
bronze_df = bronze_df.withColumn("bronze_ingest_ts", F.current_timestamp())
bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
display(spark.table(bronze_table))

## Silver cleanup

Silver enforces timestamps, removes null readings, and standardizes telemetry for monitoring logic.

In [ ]:
silver_df = (
    spark.table(bronze_table)
    .withColumn("reading_ts", F.to_timestamp("reading_ts_raw"))
    .drop("reading_ts_raw")
    .filter(F.col("device_id").isNotNull())
    .filter(F.col("reading_ts").isNotNull())
    .filter(F.col("reading_value").isNotNull())
    .withColumn("sensor_type", F.lower(F.trim("sensor_type")))
    .withColumn("unit", F.lower(F.trim("unit")))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
display(spark.table(silver_table).orderBy("device_id", "reading_ts"))

## Heartbeat monitoring

Devices that stop reporting on time are often more important operationally than single bad measurements.

In [ ]:
device_window = Window.partitionBy("device_id").orderBy("reading_ts")

heartbeat_df = (
    spark.table(silver_table)
    .withColumn("previous_reading_ts", F.lag("reading_ts").over(device_window))
    .withColumn("gap_minutes", (F.col("reading_ts").cast("long") - F.col("previous_reading_ts").cast("long")) / 60.0)
    .withColumn("heartbeat_status", F.when(F.col("gap_minutes") > 5, F.lit("late")).otherwise(F.lit("on_time")))
)

heartbeat_df.write.format("delta").mode("overwrite").saveAsTable(heartbeat_table)
display(spark.table(heartbeat_table).orderBy("device_id", "reading_ts"))

## Anomaly candidates

This is a simple rule-based example that flags suspicious temperature and vibration readings for review.

In [ ]:
anomaly_candidates_df = (
    spark.table(silver_table)
    .withColumn(
        "anomaly_reason",
        F.when((F.col("sensor_type") == "temperature") & (F.col("reading_value") > 35), F.lit("high_temperature"))
        .when((F.col("sensor_type") == "vibration") & (F.col("reading_value") > 8), F.lit("high_vibration"))
    )
    .filter(F.col("anomaly_reason").isNotNull())
)

anomaly_candidates_df.write.format("delta").mode("overwrite").saveAsTable(anomaly_table)
display(spark.table(anomaly_table).orderBy("device_id", "reading_ts"))

In [ ]:
site_summary_df = spark.sql(f"""
SELECT
  site_id,
  count(*) AS anomaly_count,
  count(distinct device_id) AS affected_devices
FROM {anomaly_table}
GROUP BY site_id
ORDER BY anomaly_count DESC
""")

display(site_summary_df)

## Production notes

- Keep raw telemetry in bronze so device issues can be replayed and reprocessed.
- Use heartbeat monitoring to detect silent device outages.
- Start with clear rules for anomaly candidates before moving to ML-based detection.
- Track site, device, unit, and event timestamps carefully because IoT quality issues are often operational, not just analytical.